# 🐍 Master Guide: Creational Design Patterns in Python
> **Single-Source Revision Guide**: Covers Pythonic implementations of Singleton, Factory Method, Abstract Factory, Builder, and Prototype.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Creational patterns abstract object instantiation mechanisms, decoupling systems from how objects are created, composed, and represented.

---
## 🧠 2. Pythonic Creational Patterns Summary

- **Singleton**: Ensures a class has only one instance (using `__new__` or module-level singleton).
- **Factory Method**: Delegates object creation to specialized subclass factory methods.
- **Builder**: Constructs complex objects step-by-step with method chaining.
- **Prototype**: Clones existing objects via `copy.deepcopy()`.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Thread-Safe Singleton via `__new__`

In [ ]:
class DatabasePool:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.connections = 10
        return cls._instance

pool1 = DatabasePool()
pool2 = DatabasePool()

print("pool1 is pool2:", pool1 is pool2)  # True! Same instance in memory

---
## 🟡 4. Part 2: Intermediate Features

### Factory Pattern (Decoupled Creation)

In [ ]:
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message: str):
        pass

class EmailNotifier(Notifier):
    def send(self, message: str):
        return f"[EMAIL] {message}"

class SMSNotifier(Notifier):
    def send(self, message: str):
        return f"[SMS] {message}"

# Factory Method:
def get_notifier(channel: str) -> Notifier:
    factories = {"email": EmailNotifier, "sms": SMSNotifier}
    if channel not in factories:
        raise ValueError(f"Unknown channel '{channel}'")
    return factories[channel]()

notifier = get_notifier("email")
print(notifier.send("System alert: Memory 95% full!"))

### Builder Pattern (Fluent API)

In [ ]:
class QueryBuilder:
    def __init__(self, table):
        self.table = table
        self._fields = "*"
        self._where = None
    
    def select(self, *fields):
        self._fields = ", ".join(fields)
        return self  # Return self for method chaining
    
    def where(self, condition):
        self._where = condition
        return self
    
    def build(self):
        sql = f"SELECT {self._fields} FROM {self.table}"
        if self._where:
            sql += f" WHERE {self._where}"
        return sql + ";"

query = QueryBuilder("users").select("id", "email", "role").where("active = 1").build()
print("Constructed SQL:", query)

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Pattern                            | Intent                                |
+------------------------------------+---------------------------------------+
| Singleton                          | Exactly one instance globally         |
| Factory                            | Decouples caller from concrete classes|
| Builder                            | Fluent step-by-step object building   |
| Prototype                          | Clones existing objects (copy.deepcopy|
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Thread-Safe Double-Checked Locking Singleton
> **Core Principle**:
> - A naive singleton creates race conditions in multithreaded environments where two threads simultaneously check `_instance is None` and both instantiate objects.
> - Double-checked locking acquires `threading.Lock()` only if the instance has not yet been initialized, preserving maximum execution performance.

In [ ]:
# Production Thread-Safe Singleton Implementation
import threading

class AppConfigRegistry:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
                    cls._instance._settings = {
                        "app_name": "Core Python Engine",
                        "environment": "Production",
                        "log_level": "INFO"
                    }
        return cls._instance

    def get(self, key, default=None):
        return self._settings.get(key, default)

    def set(self, key, value):
        self._settings[key] = value

cfg1 = AppConfigRegistry()
cfg2 = AppConfigRegistry()
cfg1.set("log_level", "DEBUG")

print("Are both config objects identical?:", cfg1 is cfg2)
print("cfg2 sees updated setting:         :", cfg2.get("log_level"))

### 2. Multi-Database Client Connector Factory
A clean Factory Method providing uniform client interfaces for MySQL, Oracle, and SQLite databases.

In [ ]:
# Database Client Factory Drill
class DBClient(ABC):
    @abstractmethod
    def execute_query(self, sql: str) -> str:
        pass

class SQLiteClient(DBClient):
    def execute_query(self, sql: str) -> str:
        return f"[SQLite In-Memory] Executed: '{sql}'"

class MySQLClient(DBClient):
    def execute_query(self, sql: str) -> str:
        return f"[MySQL Server Port 3306] Executed: '{sql}'"

class OracleClient(DBClient):
    def execute_query(self, sql: str) -> str:
        return f"[Oracle 19c Port 1521] Executed: '{sql}'"

class DatabaseFactory:
    _drivers = {
        "sqlite": SQLiteClient,
        "mysql": MySQLClient,
        "oracle": OracleClient
    }

    @classmethod
    def create_connection(cls, db_type: str) -> DBClient:
        driver = cls._drivers.get(db_type.lower())
        if not driver:
            raise ValueError(f"Unsupported database type: '{db_type}'")
        return driver()

# Testing factory dispatch
for engine in ["sqlite", "mysql", "oracle"]:
    client = DatabaseFactory.create_connection(engine)
    print(client.execute_query("SELECT * FROM EMPLOYEES"))